In [ ]:
!pip install xgboost

In [ ]:
# IMPORT DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
SEED = 42

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    DATA_PATH = "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada_final.parquet"
else:
    DATA_PATH = "../../dataset/processed/amostra_rotulada_final.parquet"

df = pd.read_parquet(DATA_PATH)

print("Dataset carregado com sucesso.")
print(f"Shape: {df.shape}")
print(f"\nDistribuição do rótulo:")
print(df["conama_status"].value_counts())

Mounted at /content/drive
Dataset carregado com sucesso.
Shape: (59896, 27)

Distribuição do rótulo:
conama_status
Atenção         29946
Adequada        20585
Não adequada     9365
Name: count, dtype: int64


## Preparação dos Dados

Para garantir a comparabilidade entre os algoritmos avaliados, foi utilizada a mesma base de dados empregada nos experimentos anteriores. Dessa forma, foram mantidas as mesmas variáveis preditoras, o mesmo conjunto de treinamento e teste e o mesmo esquema de codificação das classes.

As variáveis categóricas foram tratadas durante o pré-processamento por meio de codificação apropriada, permitindo que o modelo pudesse utilizar simultaneamente atributos numéricos e categóricos durante o treinamento.

In [ ]:
X = df[[
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "Country",
    "Waterbody Type",
    "Nitrogen (mg/l)"
]]

y = df["conama_status"]

In [ ]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Classes codificadas:")
for classe, codigo in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)):
    print(f"{classe}: {codigo}")

Classes codificadas:
Adequada: 0
Atenção: 1
Não adequada: 2


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=SEED,
    stratify=y_encoded
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (47916, 5)
Teste: (11980, 5)


In [ ]:
categorical_features = [
    "Country",
    "Waterbody Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [ ]:
model_sem_balanceamento = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            XGBClassifier(
                objective="multi:softprob",
                eval_metric="mlogloss",
                random_state=SEED,
                n_jobs=-1
            )
        )
    ]
)

model_sem_balanceamento.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('classifier',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=None, device=None,
                               early_stopping_rounds=None,
                               enable...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=-1,
                               num_parallel_tree=None, ...))])

In [ ]:
y_train_pred = model_sem_balanceamento.predict(X_train)

train_accuracy  = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average="weighted")
train_recall    = recall_score(y_train, y_train_pred, average="weighted")
train_f1        = f1_score(y_train, y_train_pred, average="weighted")
train_cm        = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("\nClassification Report:")
print(classification_report(y_train, y_train_pred))

print("Train Confusion Matrix:")
print(train_cm)

Train Accuracy:
0.9287920527589949
Train Precision:
0.9332483338468555
Train Recall:
0.9287920527589949
Train F1:
0.9292431280087334

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.99      0.96     16468
           1       0.97      0.89      0.93     23956
           2       0.81      0.93      0.87      7492

    accuracy                           0.93     47916
   macro avg       0.90      0.94      0.92     47916
weighted avg       0.93      0.93      0.93     47916

Train Confusion Matrix:
[[16235   233     0]
 [ 1004 21264  1688]
 [    0   487  7005]]


In [ ]:
y_pred = model_sem_balanceamento.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.9045909849749583

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.97      0.95      4117
           1       0.94      0.87      0.90      5990
           2       0.76      0.88      0.82      1873

    accuracy                           0.90     11980
   macro avg       0.88      0.91      0.89     11980
weighted avg       0.91      0.90      0.91     11980


Confusion Matrix:
[[4005  112    0]
 [ 288 5192  510]
 [   0  233 1640]]


## Resultados do Modelo XGBoost

Com o objetivo de validar a robustez da nova estratégia de rotulagem proposta, foi realizado um experimento adicional utilizando o algoritmo XGBoost. A intenção desta etapa não foi apenas identificar qual algoritmo apresentava a maior acurácia, mas verificar se os ganhos observados nos experimentos anteriores estavam associados à reconstrução da classe intermediária ou se dependiam exclusivamente do modelo utilizado.

Os resultados obtidos foram bastante expressivos. No conjunto de treinamento, o XGBoost alcançou:

* Acurácia: **92,88%**
* Precisão ponderada: **93,32%**
* Recall ponderado: **92,88%**
* F1-Score ponderado: **92,92%**

No conjunto de teste, os resultados permaneceram consistentes:

* Acurácia: **90,46%**
* Precisão ponderada: **91,00%**
* Recall ponderado: **90,46%**
* F1-Score ponderado: **91,00%**

A diferença entre treino e teste foi inferior a três pontos percentuais, indicando boa capacidade de generalização e ausência de sobreajuste significativo.

### Desempenho por Classe

A classe **Adequada** apresentou os melhores resultados, atingindo:

* Precisão: **93%**
* Recall: **97%**
* F1-Score: **95%**

Esses valores indicam que o modelo consegue reconhecer com elevada confiabilidade as amostras que apresentam melhores condições ambientais.

A classe **Atenção**, construída a partir das subclasses identificadas pelos modelos binários, apresentou desempenho igualmente elevado:

* Precisão: **94%**
* Recall: **87%**
* F1-Score: **90%**

Esse resultado é particularmente importante, pois a classe intermediária foi justamente a principal fonte de dificuldade nos experimentos realizados anteriormente. O elevado desempenho obtido demonstra que a reconstrução dessa categoria permitiu representar de forma mais consistente a região de transição entre águas adequadas e não adequadas.

Para a classe **Não adequada**, foram obtidos:

* Precisão: **76%**
* Recall: **88%**
* F1-Score: **82%**

Embora a precisão seja inferior às demais classes, o alto recall indica que a maior parte das amostras críticas foi corretamente identificada pelo modelo. Em aplicações de monitoramento ambiental, essa característica é desejável, pois reduz a probabilidade de corpos hídricos potencialmente problemáticos serem classificados como adequados.

### Análise da Matriz de Confusão

A matriz de confusão evidencia um comportamento coerente do modelo.

Assim como observado nos experimentos com LightGBM, praticamente não ocorreram confusões diretas entre as classes extremas:

```text
Adequada → Não adequada = 0 ocorrências
Não adequada → Adequada = 0 ocorrências
```

Esse resultado demonstra que o modelo aprendeu adequadamente a diferença entre situações claramente adequadas e claramente não adequadas.

Os erros observados concentram-se principalmente entre classes adjacentes:

```text
Adequada ↔ Atenção
Atenção ↔ Não adequada
```

Esse comportamento é esperado, uma vez que a classe **Atenção** foi concebida justamente para representar uma região intermediária entre os dois extremos de qualidade.

Em outras palavras, quando o modelo comete erros, ele tende a confundir amostras próximas da fronteira de decisão, e não amostras completamente distintas. Isso indica que a estrutura da classificação reconstruída está alinhada ao comportamento real dos dados.

### Comparação com o LightGBM

Ao comparar os resultados obtidos com os experimentos realizados utilizando LightGBM, observa-se que ambos os algoritmos apresentaram desempenho muito semelhante.

A diferença de acurácia no conjunto de teste foi mínima:

```text
LightGBM = 90,43%
XGBoost  = 90,46%
```

Apesar da pequena vantagem do XGBoost, os resultados podem ser considerados estatisticamente equivalentes para os objetivos deste estudo. Esse comportamento é particularmente relevante porque demonstra que a principal melhoria observada não foi causada pela troca de algoritmo, mas sim pela reformulação da estratégia de rotulagem.

### Interpretação Final

Os resultados obtidos reforçam a hipótese levantada ao longo dos experimentos: a principal limitação dos modelos anteriores estava relacionada à definição da classe intermediária e não à capacidade dos algoritmos de aprendizado de máquina.

A reconstrução da classe **Atenção**, baseada nas subclasses identificadas pelos modelos binários, permitiu reduzir a sobreposição entre as categorias e criar fronteiras mais consistentes entre os níveis de qualidade da água.

O fato de tanto o LightGBM quanto o XGBoost apresentarem desempenhos muito próximos após essa reformulação constitui uma forte evidência de que o ganho obtido decorre principalmente da nova estratégia de rotulagem proposta.

Dessa forma, os resultados confirmam que a utilização das subclasses para reconstruir a região intermediária produziu uma representação mais fiel da estrutura dos dados, contribuindo para uma classificação mais robusta e interpretável da qualidade da água.
